In [1]:
# ------------------------------------------
# Was tut dieses Skript?
# Dieses Skript verarbeitet Mapillary-Coverage-Daten für OSM-Highways in Deutschland.
# Es liest die Coverage-Daten für "pano" und "regular", filtert sie nach einem Coverage-Ratio >= 0.6,
# kombiniert die Ergebnisse, entfernt Duplikate (bevorzugt "pano"), und speichert die finale Tabelle als CSV.
# siehe https://github.com/vizsim/mapillary_coverage/issues/1
# ------------------------------------------

In [2]:
import pandas as pd
import geopandas as gpd

from datetime import datetime
from pathlib import Path
import json
import glob

from config import TILES_CONFIG, PROCESSING_CONFIG, MAPILLARY_CONFIG

In [3]:
# Lese Metadaten aus JSON-Dateien
files = {
    "osm_metadata.json": "osm_data_from",
    "ml_metadata.json": "ml_data_from",
}

# set target variables
osm_data_from = None
ml_data_from = None

for fname, primary_key in files.items():
    p = Path(fname)
    if not p.exists():
        print(f"{p} not found")
        continue

    with p.open("r", encoding="utf-8") as f:
        meta = json.load(f)

    if primary_key in meta:
        value = meta[primary_key]
        if primary_key == "osm_data_from":
            osm_data_from = value
        elif primary_key == "ml_data_from":
            ml_data_from = value
        #print(f"{p.name} -> {primary_key}: {value}")

# optional: show assigned values
print("osm_data_from: ", osm_data_from)
print("ml_data_from: ", ml_data_from)

osm_data_from:  2025-12-03T21:21:00Z
ml_data_from:  2025-10-18T00:00:00Z


In [4]:
# Finde alle Bundesland-spezifischen Parquet-Dateien
OUTPUT_FOLDER = PROCESSING_CONFIG["output_folder"]

pano_files = glob.glob(f"{OUTPUT_FOLDER}/DE-*_osm-highways_mp_pano_coverage_latest.parquet")
regular_files = glob.glob(f"{OUTPUT_FOLDER}/DE-*_osm-highways_mp_regular_coverage_latest.parquet")

print(f"Gefundene Pano-Dateien: {len(pano_files)}")
print(f"Gefundene Regular-Dateien: {len(regular_files)}")

# Sammle alle Bundesländer
all_pano_dfs = []
all_regular_dfs = []

# Lade alle Pano-Dateien
for file in pano_files:
    df = pd.read_parquet(file)
    all_pano_dfs.append(df)
    print(f"  Pano: {file.split('/')[-1]} → {len(df):,} Zeilen")

# Lade alle Regular-Dateien  
for file in regular_files:
    df = pd.read_parquet(file)
    all_regular_dfs.append(df)
    print(f"  Regular: {file.split('/')[-1]} → {len(df):,} Zeilen")

# Kombiniere alle Bundesländer
if all_pano_dfs:
    gdf_mp_pano = pd.concat(all_pano_dfs, ignore_index=True)
    print(f"\n✓ Gesamt Pano: {len(gdf_mp_pano):,} Zeilen")
else:
    gdf_mp_pano = pd.DataFrame(columns=["osm_id", "mp_coverage_ratio"])
    print("\n⚠️  Keine Pano-Daten gefunden!")

if all_regular_dfs:
    gdf_mp_regular = pd.concat(all_regular_dfs, ignore_index=True)
    print(f"✓ Gesamt Regular: {len(gdf_mp_regular):,} Zeilen")
else:
    gdf_mp_regular = pd.DataFrame(columns=["osm_id", "mp_coverage_ratio"])
    print("⚠️  Keine Regular-Daten gefunden!")

Gefundene Pano-Dateien: 2
Gefundene Regular-Dateien: 2
  Pano: DE-BE_osm-highways_mp_pano_coverage_latest.parquet → 187,187 Zeilen


  Pano: DE-BY_osm-highways_mp_pano_coverage_latest.parquet → 87,401 Zeilen
  Regular: DE-BE_osm-highways_mp_regular_coverage_latest.parquet → 177,189 Zeilen


  Regular: DE-BY_osm-highways_mp_regular_coverage_latest.parquet → 403,155 Zeilen

✓ Gesamt Pano: 274,588 Zeilen
✓ Gesamt Regular: 580,344 Zeilen


### pano

In [5]:

# drop all columns except osm_id and mp_coverage_ratio
gdf_mp_pano=gdf_mp_pano[["osm_id","mp_coverage_ratio"]].copy()

# filter for coverage ratio >= threshold
gdf_mp_pano_above_threshold=gdf_mp_pano[gdf_mp_pano["mp_coverage_ratio"]>=PROCESSING_CONFIG['mp_coverage_ratio_threshold']].copy()

# add new column "mapillary_coverage" with value "pano"
gdf_mp_pano_above_threshold["mapillary_coverage"] = "pano"

gdf_mp_pano_above_threshold

,osm_id,mp_coverage_ratio,mapillary_coverage
4,4045656,1.000000,pano
6,4054010,0.926417,pano
7,4054013,1.000000,pano
8,4067882,0.770549,pano
9,4067919,1.000000,pano
...,...,...,...
274583,1455393266,1.000000,pano
274584,1455397577,1.000000,pano
274585,1455535841,0.756870,pano
274586,1455755085,0.643684,pano


### regular

In [6]:

# drop all columns except osm_id and mp_coverage_ratio
gdf_mp_regular=gdf_mp_regular[["osm_id","mp_coverage_ratio"]].copy()

# filter for coverage ratio >= threshold
gdf_mp_regular_above_threshold=gdf_mp_regular[gdf_mp_regular["mp_coverage_ratio"]>=PROCESSING_CONFIG['mp_coverage_ratio_threshold']].copy()

# add new column "mapillary_coverage" with value "regular"
gdf_mp_regular_above_threshold["mapillary_coverage"] = "regular"

gdf_mp_regular_above_threshold

,osm_id,mp_coverage_ratio,mapillary_coverage
1,4045243,1.0,regular
3,4045656,1.0,regular
5,4054013,1.0,regular
6,4067883,1.0,regular
7,4067899,1.0,regular
...,...,...,...
580337,1455826720,1.0,regular
580338,1455826721,1.0,regular
580340,1455845449,1.0,regular
580341,1455845450,1.0,regular


In [7]:
# Concatenate both GeoDataFrames
both_concat=pd.concat([gdf_mp_pano_above_threshold,gdf_mp_regular_above_threshold],ignore_index=True)

both_concat["mapillary_coverage"].value_counts()

mapillary_coverage
regular    340248
pano       172265
Name: count, dtype: int64

In [8]:
## drop duplicates, keep pano over regular
both_concat = both_concat.sort_values(by="mapillary_coverage", ascending=True).drop_duplicates(subset="osm_id", keep="first")

both_concat["mapillary_coverage"].value_counts()

mapillary_coverage
regular    254980
pano       172265
Name: count, dtype: int64

In [9]:
# check for duplicates
#both_concat["osm_id"].value_counts()

In [10]:
# drop all columns except osm_id and mapillary_coverage
both_concat=both_concat[["osm_id","mapillary_coverage"]].copy()
both_concat

,osm_id,mapillary_coverage
0,4045656,pano
114838,1409148576,pano
114839,1409182527,pano
114840,1409184521,pano
114841,1409184523,pano
...,...,...
285676,1362014650,regular
285675,1362014649,regular
285674,1362014648,regular
285672,1362014643,regular


In [11]:
# Save to CSV
#both_concat.to_csv("output/germany_osm-highways_25-10-06_mp_coverage_23-01-01until25-10-07_ratio_above_06.csv",index=False)
both_concat.to_csv("output/germany_osm-highways_mp-coverage_latest.csv",index=False)




'2025-12-03'

In [12]:
# Get the current date
current_date = datetime.now().strftime("%Y-%m-%d")

# Create the content for the README file
readme_content = f"""# Mapillary Coverage per OSM Highway — Output

This folder contains the **latest** output file for *Mapillary coverage per OSM highway analysis*.

**Data created:** {current_date}<br>
**OSM data:** {osm_data_from.split('T')[0]}<br>
**Mapillary data:** {PROCESSING_CONFIG['min_capture_date']} → {ml_data_from.split('T')[0]}<br>
**Buffer distance:** {PROCESSING_CONFIG['buffer_distance']} meters<br>
**Coverage ratio threshold:** {PROCESSING_CONFIG['mp_coverage_ratio_threshold']} (60%)

Segments are considered *covered* if at least {int(PROCESSING_CONFIG['mp_coverage_ratio_threshold']*100)}% of their length falls within {PROCESSING_CONFIG['buffer_distance']} meters of Mapillary images.
"""



# Write the README file
with open("output/README.md", "w", encoding="utf-8") as readme_file:
    readme_file.write(readme_content)